# Building Multi-Agent AI Systems with LangGraph

| # | Section | Type |
|---|---------|------|
| 1 | What Are AI Agents? | Concepts |
| 2 | Single Agent to Multi-Agent | Concepts |
| 3 | Demo 1: Multi-Agent Routing | Live Code |
| 4 | Demo 2: MCP (Database Access) | Script |
| 5 | Demo 3: Remote Agent (HTTP) | Script |
| 6 | Homework | Assignment |

---
# Section 1: What Are AI Agents?

A **chatbot** takes input, returns output. One turn.

An **agent** can: **Reason**, **Use tools**, **Remember context**, **Decide when done**.

Agent = LLM + Loop: **think → act → observe → repeat**

### Agent Components

| Component | What It Does | LangGraph Equivalent |
|-----------|-------------|---------------------|
| **Instructions** | System prompt | `prompt=` in `create_react_agent` |
| **Model** | LLM engine | `ChatOpenAI(model=...)` |
| **Tools** | Functions it can call | `tools=[...]` in `create_react_agent` |
| **Memory** | What it remembers | `MessagesState` / `MemorySaver` checkpointer |

### The Agent Loop (ReAct)

```
User Input → REASON (LLM) → ACT (Tools) → OBSERVE (Results) → Done? → Final Response
                                                                 No  → back to REASON
```

In LangGraph, this loop is what `create_react_agent` implements internally.

---
# Section 2: From Single Agent to Multi-Agent

### The Problem: "God Agent"
One agent with 15+ tools, a 2000-word prompt, contradicting instructions. Tool selection degrades.

### The Solution: Split the Work

```
        Supervisor Node (LLM Router)
          /         |          \
    billing_agent  technical_agent  escalation_agent
```

Each agent: focused job, short `prompt=`, limited tools.

| Benefit | Description |
|---------|-------------|
| Modularity | Change one without breaking others |
| Specialization | Fewer tools = better tool selection |
| Testability | Test each agent in isolation |
| Scalability | Add agents, don't bloat existing ones |

### LangGraph Multi-Agent Patterns

| Pattern | LangGraph | ADK Equivalent |
|---------|-----------|----------------|
| **Router / Delegation** | Supervisor + `Command(goto=...)` | `Agent(sub_agents=[...])` |
| **Sequential Pipeline** | `add_edge(A, B)`, `add_edge(B, C)` | `SequentialAgent` |
| **Parallel Fan-Out** | `Send` API | `ParallelAgent` |
| **Loop / Iterative** | Conditional edge back to same node | `LoopAgent` |

---
# Section 3: LangGraph — Demo 1

**LangGraph** — LangChain's framework for building stateful, multi-actor agent systems using a graph abstraction.

| Concept | LangGraph | ADK |
|---------|-----------|-----|
| Agent with tools | `create_react_agent(llm, tools=[...])` | `Agent(tools=[...])` |
| Multi-agent routing | `StateGraph` + supervisor `Command` | `Agent(sub_agents=[...])` |
| Sequential pipeline | `add_edge(A, B)` | `SequentialAgent` |
| Parallel fan-out | `Send` API | `ParallelAgent` |
| LLM-driven routing | `llm.with_structured_output(Router)` | Automatic via `description=` |

### Demo 1 Architecture

```
START → Supervisor (Router LLM)
          |--- billing_agent    → lookup_invoice, process_refund → END
          |--- technical_agent  → search_knowledge_base, check_system_status → END
          |--- escalation_agent → create_escalation_ticket → END
```

In [1]:
# Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "langgraph", "langchain", "langchain-core", "langchain-google-genai",
    "langchain-mcp-adapters", "python-dotenv", "httpx", "langfuse", "-q"])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.43.0 requires opentelemetry-exporter-otlp-proto-common==1.43.0, but you have opentelemetry-exporter-otlp-proto-common 1.44.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.43.0 requires opentelemetry-proto==1.43.0, but you have opentelemetry-proto 1.44.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.43.0 requires opentelemetry-sdk~=1.43.0, but you have opentelemetry-sdk 1.44.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


0

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command
from pydantic import BaseModel
from typing import Literal

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0)

print("Ready" if os.getenv("OPENAI_API_KEY") else "WARNING: Set OPENAI_API_KEY in .env")

Ready


### Helper: Extract Final Response

In [3]:
def get_response(messages: list) -> str:
    """Extract the last substantive AI response from a message list."""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content and not getattr(msg, "tool_calls", None):
            return msg.content
    return "(no response)"

def ask(graph, message: str) -> str:
    """Run a compiled LangGraph graph with a single user message."""
    result = graph.invoke({"messages": [HumanMessage(content=message)]})
    return get_response(result["messages"])

### Step 1: Define Tools

Tools are plain Python functions with docstrings. The LLM reads the docstring to decide when and how to call them.

In [4]:
# Billing tools

def lookup_invoice(customer_email: str) -> dict:
    """Look up the most recent invoice for a customer by email."""
    invoices = {
        "jane@example.com": {"invoice_id": "INV-2024-001", "customer": "Jane Doe", "amount": "$99.00", "status": "paid", "plan": "Pro Monthly"},
        "bob@example.com": {"invoice_id": "INV-2024-002", "customer": "Bob Smith", "amount": "$299.00", "status": "overdue", "plan": "Enterprise Annual"},
        "alice@example.com": {"invoice_id": "INV-2024-003", "customer": "Alice Johnson", "amount": "$49.00", "status": "paid", "plan": "Starter Monthly"},
    }
    return invoices.get(customer_email.lower(), {"error": f"No invoice found for {customer_email}"})

def process_refund(invoice_id: str, reason: str) -> dict:
    """Process a refund for a specific invoice."""
    return {"refund_id": f"REF-{invoice_id[-3:]}", "status": "approved", "message": f"Refund for {invoice_id} approved. 5-7 business days."}

# Technical tools

def search_knowledge_base(query: str) -> dict:
    """Search the knowledge base for technical solutions."""
    articles = {
        "login": {"title": "Login Issues", "solution": "1. Clear cache. 2. Try incognito. 3. Reset password."},
        "crash": {"title": "App Crashing", "solution": "1. Update to v3.2.1. 2. Clear app data. 3. Check OS requirements."},
        "slow": {"title": "Performance Issues", "solution": "1. Check internet. 2. Close other apps. 3. Enable hardware acceleration."},
    }
    for keyword, article in articles.items():
        if keyword in query.lower():
            return article
    return {"title": "General Support", "solution": "No specific article found."}

def check_system_status() -> dict:
    """Check current status of all platform services."""
    return {"overall": "operational", "auth_service": "degraded", "last_incident": "2026-02-08"}

# Escalation tool

def create_escalation_ticket(customer_email: str, issue_summary: str, priority: str) -> dict:
    """Create an escalation ticket for issues needing human review."""
    times = {"low": "48 hours", "medium": "24 hours", "high": "4 hours", "critical": "1 hour"}
    return {"ticket_id": "ESC-2026-0042", "priority": priority, "estimated_response": times.get(priority, "24 hours")}

### Step 2: Create Specialist Agents

Each specialist is a `create_react_agent` — it runs a ReAct loop with its own tools and system prompt.

Compare with ADK:
```python
# ADK
billing_agent = Agent(name="billing_agent", model=MODEL, 
                      instruction="...", tools=[lookup_invoice, process_refund])

# LangGraph
billing_react = create_react_agent(llm, tools=[lookup_invoice, process_refund], prompt="...")
```

In [5]:
def _make_specialist(tools: list, system_prompt: str):
    """Wrap a create_react_agent as a graph node that returns only new messages."""
    agent = create_react_agent(llm, tools=tools, prompt=system_prompt)
    def node(state: MessagesState):
        result = agent.invoke(state)
        return {"messages": result["messages"][len(state["messages"]):]}  # new msgs only
    return node

billing_node = _make_specialist(
    tools=[lookup_invoice, process_refund],
    system_prompt="You are a billing specialist. Use lookup_invoice and process_refund.",
)

technical_node = _make_specialist(
    tools=[search_knowledge_base, check_system_status],
    system_prompt="You are a technical specialist. Use search_knowledge_base and check_system_status.",
)

escalation_node = _make_specialist(
    tools=[create_escalation_ticket],
    system_prompt="You are an escalation specialist. Use create_escalation_ticket to log cases.",
)

/var/folders/4z/ycq9kt293wq0pq0hwvd979gr0000gn/T/ipykernel_50356/3128566217.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools, prompt=system_prompt)
/var/folders/4z/ycq9kt293wq0pq0hwvd979gr0000gn/T/ipykernel_50356/3128566217.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools, prompt=system_prompt)
/var/folders/4z/ycq9kt293wq0pq0hwvd979gr0000gn/T/ipykernel_50356/3128566217.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to 

### Step 3: Create the Supervisor (Router)

The supervisor uses **structured output** to decide which node to route to.

Compare with ADK:
```python
# ADK: routing is automatic based on sub-agent description
root_agent = Agent(sub_agents=[billing_agent, technical_agent, escalation_agent])

# LangGraph: explicit structured-output routing
class Router(BaseModel):
    next: Literal["billing_agent", "technical_agent", "escalation_agent", "FINISH"]

def supervisor_node(state) -> Command:
    response = llm.with_structured_output(Router).invoke(...)
    return Command(goto=response.next)
```

**Trade-off:** ADK routing is implicit (magic); LangGraph routing is explicit (transparent).

In [6]:
class Router(BaseModel):
    """Routing decision — which specialist should handle this request?"""
    next: Literal["billing_agent", "technical_agent", "escalation_agent", "FINISH"]

SUPERVISOR_PROMPT = (
    "You are a customer support router. Decide which specialist should handle this:\n"
    "- billing_agent: invoices, payments, refunds\n"
    "- technical_agent: bugs, crashes, performance, system status\n"
    "- escalation_agent: complaints, disputes, fraud, security\n"
    "Output FINISH when the conversation is already resolved."
)

def supervisor_node(state: MessagesState) -> Command:
    response = llm.with_structured_output(Router).invoke(
        [SystemMessage(content=SUPERVISOR_PROMPT)] + state["messages"]
    )
    goto = "__end__" if response.next == "FINISH" else response.next
    print(f"  [Supervisor] → routing to: {goto}")
    return Command(goto=goto)

### Step 4: Wire the Graph

Compare with ADK:
```python
# ADK: Runner handles execution
runner = Runner(agent=root_agent, ...)

# LangGraph: compile the graph → call invoke()
app = builder.compile()
result = app.invoke({"messages": [...]})
```

In [7]:
builder = StateGraph(MessagesState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("billing_agent", billing_node)
builder.add_node("technical_agent", technical_node)
builder.add_node("escalation_agent", escalation_node)

builder.add_edge(START, "supervisor")
# Specialists go directly to END (no loop back)
builder.add_edge("billing_agent", END)
builder.add_edge("technical_agent", END)
builder.add_edge("escalation_agent", END)

routing_app = builder.compile()
print("Graph compiled successfully")

Graph compiled successfully


### Step 5: Test It

In [8]:
# Test 1: Billing
response = ask(routing_app, "I need to check my invoice. Email: bob@example.com. Can I get a refund?")
print(response)

  [Supervisor] → routing to: billing_agent
I found your invoice with the following details:

- **Invoice ID:** INV-2024-002
- **Customer Name:** Bob Smith
- **Amount:** $299.00
- **Status:** Overdue
- **Plan:** Enterprise Annual

Would you like to proceed with processing a refund? If so, please provide a reason for the refund.


In [9]:
# Test 2: Technical
response = ask(routing_app, "My app keeps crashing when I login. Is there an outage?")
print(response)

  [Supervisor] → routing to: technical_agent
The current system status indicates that all services are operational, but the authentication service is experiencing degraded performance. This could be related to the issues you're facing with the app crashing during login.

Here are some suggested solutions to try:

1. **Clear Cache**: Clear the app's cache to remove any corrupted data.
2. **Try Incognito Mode**: If you're using a web app, try logging in through an incognito or private browsing window.
3. **Reset Password**: If the issue persists, consider resetting your password.

If the problem continues, it may be worth checking back later as the authentication service is currently not fully operational.


In [10]:
# Test 3: Escalation
response = ask(routing_app, "Someone hacked my account! I see charges I didn't make. Email: jane@example.com. Urgent!")
print(response)

  [Supervisor] → routing to: escalation_agent
I've logged your case as an urgent escalation ticket. Here are the details:

- **Ticket ID:** ESC-2026-0042
- **Priority:** High
- **Estimated Response Time:** 4 hours

Please keep an eye on your email for updates.


---
# Section 4: MCP — Demo 2

**MCP (Model Context Protocol)** — open standard for agents to connect to external tools/data at runtime.

```
create_react_agent → MultiServerMCPClient → Supabase MCP Server (npx) → Postgres DB
```

The agent **discovers tools at runtime** — no hardcoded SQL or table definitions.

### Key Code

```python
# ADK
supabase_mcp = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@supabase/mcp-server-supabase@latest", "--access-token", TOKEN],
        )
    )
)
billing_agent = Agent(tools=[supabase_mcp])

# LangGraph (v0.1.0+: no context manager, get_tools() is now async)
client = MultiServerMCPClient({
    "supabase": {"command": "npx", "args": [...], "transport": "stdio"}
})
tools = await client.get_tools()
agent = create_react_agent(llm, tools=tools)
result = await agent.ainvoke({"messages": [HumanMessage(content=msg)]})
```

### MCP vs Hardcoded Tools

| | Demo 1 (Hardcoded) | Demo 2 (MCP) |
|---|---|---|
| Data | Fake dictionaries | Real Supabase DB |
| Tools | Manual function defs | Auto-discovered at runtime |
| Queries | Fixed in code | Agent writes its own SQL |

In [11]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

TOKEN = os.getenv("SUPABASE_ACCESS_TOKEN", "")
PROJECT_REF = os.getenv("SUPABASE_PROJECT_REF", "")

if not TOKEN:
    print("WARNING: SUPABASE_ACCESS_TOKEN not set")
else:
    print(f"Supabase token: {TOKEN[:10]}...  Project: {PROJECT_REF}")

Supabase token: sbp_30ac56...  Project: kbgzwiumhxbpcmkwqvjo


In [12]:
async def ask_mcp(message: str) -> str:
    """Run a query via MCP-connected Supabase agent."""
    args = ["-y", "@supabase/mcp-server-supabase@latest", "--access-token", TOKEN]
    if PROJECT_REF:
        args += ["--project-ref", PROJECT_REF]

    # langchain-mcp-adapters 0.1.0+: no context manager, get_tools() is async
    client = MultiServerMCPClient({"supabase": {"command": "npx", "args": args, "transport": "stdio"}})
    tools = await client.get_tools()
    print(f"  [MCP] Discovered {len(tools)} tools: {[t.name for t in tools[:5]]}...")

    agent = create_react_agent(
        llm, tools=tools,
        prompt="You are a billing specialist. Use MCP tools to query customers, orders, support_tickets.",
    )
    result = await agent.ainvoke({"messages": [HumanMessage(content=message)]})
    return get_response(result["messages"])

In [13]:
# Test MCP: Customer Lookup (requires Supabase credentials)
# Launches npx @supabase/mcp-server-supabase as a subprocess
response = await ask_mcp("What orders does Bob Smith have? What's the total amount?")
print(response)

  [MCP] Discovered 20 tools: ['search_docs', 'list_tables', 'list_extensions', 'list_migrations', 'apply_migration']...


/var/folders/4z/ycq9kt293wq0pq0hwvd979gr0000gn/T/ipykernel_50356/1758773232.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Bob Smith has the following orders:

1. Order ID: ORD-1002, Amount: $2999.00
2. Order ID: ORD-1009, Amount: $499.00

The total amount for all orders is $3498.00.


In [14]:
# Test MCP: Cross-table Query
response = await ask_mcp("Show me all high-priority open support tickets with customer name and email.")
print(response)

  [MCP] Discovered 20 tools: ['search_docs', 'list_tables', 'list_extensions', 'list_migrations', 'apply_migration']...


/var/folders/4z/ycq9kt293wq0pq0hwvd979gr0000gn/T/ipykernel_50356/1758773232.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Here are the high-priority open support tickets along with the customer names and emails:

1. **Customer Name:** Jane Doe
   - **Email:** jane@example.com

2. **Customer Name:** James Anderson
   - **Email:** james@example.com

3. **Customer Name:** Maria Garcia
   - **Email:** maria@example.com

4. **Customer Name:** Tom White
   - **Email:** tom@example.com


---
# Section 5: Remote Agent — Demo 3

**ADK A2A vs LangGraph HTTP** — both enable cross-service agent communication.

```
ADK:       Root Agent  ---A2A (HTTP/JSON-RPC)---> Shipping Agent (:8001)
LangGraph: Main Graph  ---HTTP POST /invoke-----> FastAPI Agent  (:8001)
```

### Key Code

```python
# ADK Server
app = to_a2a(shipping_agent, port=8001)  # custom A2A protocol

# LangGraph Server (shipping_agent.py)
@app.post("/invoke")
async def invoke(req: InvokeRequest):
    result = shipping_react_agent.invoke({"messages": [HumanMessage(content=req.message)]})
    return {"response": get_response(result["messages"])}

# ---

# ADK Client
shipping = RemoteA2aAgent(name="shipping", agent_card="http://localhost:8001")

# LangGraph Client (node in main graph)
async def shipping_node(state: MessagesState):
    user_msg = next(m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage))
    async with httpx.AsyncClient() as client:
        resp = await client.post("http://localhost:8001/invoke", json={"message": user_msg})
    return {"messages": [AIMessage(content=resp.json()["response"], name="shipping_agent")]}
```

| | ADK A2A | LangGraph HTTP |
|---|---|---|
| Protocol | Custom A2A JSON-RPC | Plain REST + JSON |
| Discovery | Agent Card JSON | Simple GET endpoint |
| Transport | HTTP | HTTP |
| Complexity | Higher (protocol overhead) | Lower (just httpx) |

### Run It (two terminals)

**Terminal 1:** `uvicorn shipping_agent:app --port 8001`

**Terminal 2:** `python demo3_full_system.py`

In [15]:
import httpx

def check_shipping_agent(url="http://localhost:8001") -> bool:
    try:
        resp = httpx.get(f"{url}/.well-known/agent-card.json", timeout=2)
        return resp.status_code == 200
    except Exception:
        return False

if check_shipping_agent():
    print("Shipping agent is running at :8001")
else:
    print("Shipping agent NOT running — start with: uvicorn shipping_agent:app --port 8001")

Shipping agent is running at :8001


In [16]:
# Test shipping agent directly via HTTP
async def ask_shipping(message: str) -> str:
    async with httpx.AsyncClient(timeout=30.0) as client:
        resp = await client.post("http://localhost:8001/invoke", json={"message": message})
        return resp.json()["response"]

# Run only if agent is running
if check_shipping_agent():
    response = await ask_shipping("Where is my package for order ORD-1004? When will it arrive?")
    print(response)
else:
    print("Start the shipping agent first")

Your package for order **ORD-1004** is currently with **DHL** and is **out for delivery** from a local facility. 

It is expected to arrive today, **February 13, 2026**, between **12 PM and 4 PM**.


---
# Full System Architecture

All three layers combined: routing + MCP + remote agent.

```
        Supervisor Node (Router LLM)
       /           |             \
  billing_agent  technical_agent  shipping_node
   (async MCP)     (local)         (HTTP POST)
       |             |                 |
   Supabase DB   KB/Status        FastAPI :8001
```

```python
# ADK full system
root_agent = Agent(sub_agents=[
    billing_agent,    # McpToolset -> Supabase
    technical_agent,  # local tools
    RemoteA2aAgent(agent_card="http://localhost:8001"),  # A2A
])

# LangGraph full system
builder = StateGraph(MessagesState)
builder.add_node("supervisor", supervisor_node)       # routes via Command
builder.add_node("billing_agent", billing_node)       # async def, MCP tools
builder.add_node("technical_agent", technical_node)   # local tools
builder.add_node("shipping_agent", shipping_node)     # HTTP POST to :8001
```

### Run It

```bash
# Terminal 1
uvicorn shipping_agent:app --port 8001

# Terminal 2
python demo3_full_system.py
```

### Interactive Demo

```bash
streamlit run streamlit_app.py
```

### Project Files

| File | What | Run |
|------|------|-----|
| `demo1_routing.py` | Multi-agent routing | `python demo1_routing.py` |
| `demo2_mcp.py` | MCP + Supabase | `python demo2_mcp.py` |
| `demo3_full_system.py` | Full system | `python demo3_full_system.py` |
| `shipping_agent.py` | Remote FastAPI agent | `uvicorn shipping_agent:app --port 8001` |
| `streamlit_app.py` | Interactive demo | `streamlit run streamlit_app.py` |

---
# Section 6: Homework

Build your own multi-agent system using **LangGraph**.

### Requirements

1. 3+ specialist agents with distinct responsibilities
2. A supervisor node that routes via `Command(goto=...)`
3. At least one MCP integration (`MultiServerMCPClient`)
4. At least one remote agent node (HTTP call)

### Suggested Domains

| Domain | Agents |
|--------|--------|
| E-commerce | Product search, orders, customer service, inventory |
| Healthcare | Symptom checker, appointments, prescriptions |
| DevOps | Log analyzer, deployment manager, incident responder |
| Travel | Flights, hotels, itinerary, local recommendations |

### Deliverables

1. Working code with a supervisor + 3 specialists
2. At least one standalone FastAPI agent (equivalent of A2A)
3. Brief writeup: architecture, which agents use MCP vs HTTP, one challenge

### Stretch Goals

- Sequential pipeline: `add_edge(draft_node, review_node)`, `add_edge(review_node, edit_node)`
- Parallel fan-out with `Send` API
- Loop until quality met with a conditional edge
- Add `MemorySaver` checkpointer for multi-turn conversations

---

## Summary

| Section | Key Concept | LangGraph API |
|---------|-------------|---------------|
| 1 | Agent = LLM + Tools + Loop | `create_react_agent` |
| 2 | Split work, not prompts | `StateGraph` nodes |
| 3 | Explicit structured routing | `llm.with_structured_output(Router)` + `Command(goto=...)` |
| 4 | MCP auto-discovers tools | `MultiServerMCPClient` |
| 5 | HTTP replaces A2A | `httpx.AsyncClient().post(...)` |

```
Layer 1: Multi-Agent → Split god-agent into specialists (StateGraph + supervisor)
Layer 2: MCP         → Connect agents to real data (MultiServerMCPClient)
Layer 3: Remote      → Connect agents to other services (httpx HTTP POST)
```

## Resources

- [LangGraph Docs](https://langchain-ai.github.io/langgraph/) | [LangGraph GitHub](https://github.com/langchain-ai/langgraph)
- [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)
- [LangGraph Multi-Agent Supervisor Tutorial](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/agent_supervisor/)
- [Google AI Studio (get API key)](https://aistudio.google.com/apikey)